# Python Best Practices


## 1. Naming (PEP 8)

Python's style guide, **PEP 8**, defines naming conventions almost every Python codebase follows:

| What | Convention | Example |
|---|---|---|
| Variables & functions | `snake_case` | `total_price`, `calculate_total()` |
| Classes | `PascalCase` | `Book`, `LibraryManager` |
| Constants | `UPPER_SNAKE_CASE` | `MAX_BOOKS_PER_MEMBER = 5` |
| "Private" attributes | leading underscore | `self._internal_cache` |

Consistent naming means anyone reading your code (including future you) can tell what a name
*is* just by looking at it.

In [1]:
# Not Pythonic
class book:
    def __init__(self, Title, Author):
        self.Title = Title
        self.Author = Author

MaxBooks = 5

# Pythonic
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

MAX_BOOKS = 5

print("Naming comparison shown above (not executed for effect).")

Naming comparison shown above (not executed for effect).


## 2. Readability over cleverness

Code is read far more often than it's written. Prefer the obvious solution over the clever
one-liner, and use **docstrings** to explain *why*, not just *what*.

In [2]:
def calculate_late_fee(days_late, daily_rate=0.50):
    """Return the late fee for a book, given days overdue and a daily rate.

    Returns 0 if the book is not overdue.
    """
    if days_late <= 0:
        return 0
    return days_late * daily_rate


print(calculate_late_fee(3))
print(calculate_late_fee(0))

1.5
0


**Guideline:** if you need a comment to explain *what* a line does, consider renaming
variables or splitting the line instead. Save comments for explaining *why* a decision was made.

## 3. Function and class design

- **Single responsibility** — a function should do one thing. If you're using "and" to describe
  what a function does (e.g. "validates input *and* saves to file"), it's probably two functions.
- **Keep functions short** — if a function needs a lot of scrolling to read, it's a candidate for
  splitting.
- **Avoid deeply nested code** — more than 2–3 levels of nested `if`/`for` is a sign to extract a
  helper function or use early returns.
- **Avoid global state** — pass data in as parameters and return results, rather than reading and
  writing variables defined outside any function.

In [12]:
# Harder to read: deep nesting
def process_book_deep(book, member):
    if book is not None:
        if not book.is_borrowed:
            if member is not None:
                book.is_borrowed = True
                return True
    return False


# Easier to read: early returns ("guard clauses")
def process_book_flat(book, member):
    if book is None or member is None:
        return False
    if book.is_borrowed:
        return False
    book.is_borrowed = True
    return True


print(process_book_deep(None, None))
print(process_book_flat(None, None))

False
False


## 4. Pythonic idioms

A few small habits that make Python code look and read like Python, not like another language
translated into Python syntax.

In [13]:
name = "Ahmed"
age = 30

# Not Pythonic
message_old = "My name is " + name + " and I am " + str(age) + " years old."

# Pythonic — f-strings
message_new = f"My name is {name} and I am {age} years old."

print(message_old)
print(message_new)

My name is Ahmed and I am 30 years old.
My name is Ahmed and I am 30 years old.


In [5]:
numbers = [1, 2, 3, 4, 5, 6]

# Not Pythonic
squares_old = []
for n in numbers:
    if n % 2 == 0:
        squares_old.append(n ** 2)

# Pythonic — list comprehension
squares_new = [n ** 2 for n in numbers if n % 2 == 0]

print(squares_old)
print(squares_new)

[4, 16, 36]
[4, 16, 36]


In [6]:
names = ["Ahmed", "Mona", "Youssef"]

# Not Pythonic
i = 0
while i < len(names):
    print(i, names[i])
    i += 1

print()

# Pythonic — enumerate()
for index, name in enumerate(names):
    print(index, name)

0 Ahmed
1 Mona
2 Youssef

0 Ahmed
1 Mona
2 Youssef


In [7]:
names = ["Ahmed", "Mona", "Youssef"]
ages = [30, 25, 28]

# Not Pythonic
for i in range(len(names)):
    print(names[i], ages[i])

print()

# Pythonic — zip()
for name, age in zip(names, ages):
    print(name, age)

Ahmed 30
Mona 25
Youssef 28

Ahmed 30
Mona 25
Youssef 28


In [8]:
# Always use a `with` block for files — it closes the file automatically,
# even if an error happens inside the block.

with open("practices_demo.txt", "w") as file:
    file.write("Best practices demo\n")

with open("practices_demo.txt", "r") as file:
    print(file.read())

Best practices demo



## 5. Error handling discipline

- Catch **specific** exceptions (`ValueError`, `FileNotFoundError`, a custom exception) — never a
  bare `except:`, which silently swallows *everything*, including typos and `KeyboardInterrupt`.
- Fail loudly while developing; only handle errors gracefully at the boundaries where the user
  interacts with your program (like the menu loop from Day 6).
- Use custom exceptions to make error handling self-documenting, as you did with
  `BookNotFoundError`, `MemberNotFoundError`, etc.

In [9]:
# Not recommended — swallows every possible error, including bugs
def risky_bare_except():
    try:
        return 10 / 0
    except:
        return None


# Recommended — catches only what you expect, and re-raises everything else
def risky_specific_except():
    try:
        return 10 / 0
    except ZeroDivisionError:
        print("Handled: division by zero.")
        return None


print(risky_bare_except())
print(risky_specific_except())

None
Handled: division by zero.
None


## 6. Project hygiene

Habits that matter once a project grows beyond a single notebook or script:

- **Virtual environments** — use `python -m venv venv` per project so dependencies don't clash
  between projects.
- **`requirements.txt`** — record your project's dependencies (`pip freeze > requirements.txt`)
  so anyone (including future you) can recreate your environment.
- **Split code into modules** — as in Day 5, keep `models.py` (classes), `storage.py`
  (save/load), and `main.py` (entry point / menu loop) separate rather than one giant file.
- **Version control (git)** — even for small solo projects, `git init` + regular commits gives
  you a history you can roll back to, and is essential the moment you collaborate with anyone
  else.
- **Consistent structure** — a predictable folder layout (e.g. `project/`, `project/models.py`,
  `project/main.py`, `project/data/`) makes a project easier to navigate months later.

## 7. Before / after — putting it all together

A single example applying several of the practices above at once.

In [10]:
# Before
def f(l):
    r = []
    for i in l:
        if i["is_borrowed"] == False:
            r.append(i["title"])
    return r


books = [
    {"title": "1984", "is_borrowed": False},
    {"title": "Brave New World", "is_borrowed": True},
]
print(f(books))

['1984']


In [11]:
# After
def get_available_titles(books):
    """Return the titles of all books that are not currently borrowed."""
    return [book["title"] for book in books if not book["is_borrowed"]]


print(get_available_titles(books))

['1984']


What changed, and why it matters:
- `f` / `l` / `r` → `get_available_titles` / `books` / return value: names now describe intent.
- Manual loop + `.append()` → list comprehension: shorter and more idiomatic.
- `== False` → `not book["is_borrowed"]`: avoids the classic beginner comparison-to-boolean
  anti-pattern.
- Added a docstring: the function's purpose is clear without reading its body.

---
## Closing

That's the full six-day arc: **fundamentals → data structures → OOP → error handling & files →
a real project → the habits that make your code professional.**

The best way to keep these habits is to apply them the next time you touch your Library
Management System — or any new project. Good luck, and happy coding!

### Thank You
Python Programming — Course Closer: Best Practices